# LOGISTPULSE · Teoría de la Computación sobre la operación del lakehouse

**Pregunta de fondo:** de todos los problemas que aparecen al despachar, ¿cuáles podemos resolver bien y rápido (**P**), cuáles solo sabemos **verificar** rápido (**NP**), y qué garantía puede prometer el negocio en cada caso?

Cada sección sigue el mismo guion de teoría: **problema → clase → certificado y verificador → algoritmo exacto → heurística y su garantía → qué tamaño aguanta (medido) → lectura para la gerencia.**

> Los datos salen del **lakehouse** si está disponible (`ods`, `master`, `dw`); si no, se usan datos de **ejemplo con la misma forma** y el notebook lo dice. El lakehouse es sintético (`lp.seed`): ilustra el método, no representa a DANEC.

In [ ]:
import sys, time, pathlib
# Localiza el código del proyecto tanto en el repo como en el contenedor Jupyter
here = pathlib.Path.cwd().resolve()
for base in [here, *here.parents, pathlib.Path('/')]:
    for cand in (base / 'deberes' / 'teoria-computacion', base / 'teoria'):
        if (cand / 'np_lab').exists() and str(cand) not in sys.path: sys.path.insert(0, str(cand))
    for cand in (base / 'base' / 'agent', base / 'agent'):
        if (cand / 'lpagent').exists() and str(cand) not in sys.path: sys.path.insert(0, str(cand))
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
from np_lab import datos, logistica as L, registro as reg, tsp
from lpagent.grid import DEFAULT_LAYOUT as GRID, distance_matrix

FUENTE = 'auto'          # 'lakehouse' (exige la base), 'ejemplo' o 'auto'
op = datos.cargar(FUENTE, n=24)
S = datos.SUP
MPC = S['minutos_por_unidad_de_costo']
runs = []                 # aquí se acumulan las corridas (se guardan al final)
print('Fuente:', op.fuente, '|', op.deposito, '| pedidos pendientes:', len(op.pedidos))
if op.aviso: print('AVISO:', op.aviso)

## 1. Datos de partida y la clase **P**: las distancias

Cada pedido pendiente (`ods.orders` en `CREATED`) es una parada. **Supuesto declarado:** la base no tiene coordenadas, así que cada cliente se ubica en una celda del plano del almacén (ancla de su zona + desplazamiento determinista; ver `np_lab/supuestos_teoria.json`).

El costo entre dos paradas es el camino mínimo en la grilla: **Dijkstra, O((V+E) log V), problema en P**. Es lo único de esta parte que escala sin sufrir.

In [ ]:
pedidos = op.pedidos
pts = [(0, 0)] + [datos.celda(p['customer_id'], p['zone_id'], GRID) for p in pedidos]
D = reg.medir(runs, op.fuente, 'tsp', 'dijkstra-matriz', len(pedidos), True, lambda: distance_matrix(GRID, pts))
t = runs[-1].segundos
df = pd.DataFrame(pedidos)
print(f'Matriz {len(D)}x{len(D)} calculada en {t*1000:.1f} ms (P).')
display(df.groupby('segment').agg(pedidos=('order_id','count'), kg_prom=('kg','mean'), restante_prom_min=('restante_min','mean')).round(1))

## 2. Lo que hace a un problema **NP**: verificar es fácil, encontrar no

Para el recorrido, el **certificado** es el orden de las paradas. El verificador suma costos y compara con el presupuesto: **O(n)**. Encontrar el mejor orden es otra historia. Midamos la diferencia con las mismas paradas.

In [ ]:
k = 9
sub = [[D[a][b] for b in range(k+1)] for a in range(k+1)]
tour_hk, c_hk = tsp.held_karp(sub)
budget = c_hk
t0 = time.perf_counter(); ok = tsp.verify_tour(sub, budget, tour_hk); t_ver = time.perf_counter() - t0
t0 = time.perf_counter(); bf = tsp.brute_force(sub, budget); t_bf = time.perf_counter() - t0
print(f'n={k}: verificar un certificado: {t_ver*1e6:.0f} µs  |  buscarlo por fuerza bruta: {t_bf:.3f} s  |  ratio ≈ {t_bf/max(t_ver,1e-9):,.0f}×')
assert ok

## 3. TSP-decisión (el recorrido de un viaje): **NP-completo**

¿Existe un ciclo desde el depósito que visite las $n$ paradas con costo ≤ presupuesto? Algoritmos: fuerza bruta $O(n!)$, **Held-Karp** $O(n^2 2^n)$ (exacto), vecino más cercano y **2-opt** (heurísticos, sin garantía). Medimos con los pedidos reales (o de ejemplo), tomando las $n$ primeras paradas.

In [ ]:
filas = []
for n in range(5, 13):
    sub = [[D[a][b] for b in range(n+1)] for a in range(n+1)]
    tour_h, c_h = reg.medir(runs, op.fuente, 'tsp', 'held-karp', n, True, lambda: tsp.held_karp(sub), valor_de=lambda r: r[1])
    nn = reg.medir(runs, op.fuente, 'tsp', 'vecino-mas-cercano', n, False, lambda: tsp.nearest_neighbor(sub),
                   optimo=c_h, valor_de=lambda r: tsp.tour_cost(sub, r))
    o2 = reg.medir(runs, op.fuente, 'tsp', 'nn+2opt', n, False, lambda: tsp.two_opt(sub, tsp.nearest_neighbor(sub)),
                   optimo=c_h, valor_de=lambda r: tsp.tour_cost(sub, r))
    fila = {'n': n, 'óptimo': c_h, 'NN': tsp.tour_cost(sub, nn), 'NN+2opt': tsp.tour_cost(sub, o2)}
    if n <= 9:
        reg.medir(runs, op.fuente, 'tsp', 'fuerza-bruta', n, True, lambda: tsp.brute_force(sub))
    filas.append(fila)
tabla = pd.DataFrame(filas); tabla['brecha NN %'] = (100*(tabla['NN']-tabla['óptimo'])/tabla['óptimo']).round(1)
display(tabla)
r = pd.DataFrame([x.__dict__ for x in runs if x.problema=='tsp' and x.algoritmo in ('held-karp','fuerza-bruta')])
for alg, g in r.groupby('algoritmo'): plt.semilogy(g['n'], g['segundos'], marker='o', label=alg)
plt.xlabel('paradas n'); plt.ylabel('segundos (log)'); plt.title('Exacto: fuerza bruta vs Held-Karp'); plt.legend(); plt.show()

**Lectura.** Held-Karp sigue siendo exponencial, pero para $n \le 12$ (el tope del validador del agente) responde en milisegundos: **un problema NP-completo puede ser práctico si $n$ es chico y está acotado**. La heurística pierde algunos puntos de costo; su "no" **no es una prueba** de que no exista un ciclo.

## 4. Mochila 0/1: qué pedidos suben a un viaje (**NP-completo… pero pseudo-polinomial**)

Capacidad de un vehículo (supuesto: `capacidad_kg`), peso por pedido (`order_items × product.weight_kg`) y valor = prioridad del segmento. Con pesos enteros, la **programación dinámica** resuelve en $O(n \cdot W)$: polinomial en el *valor* de $W$, **exponencial en su tamaño en bits**. Veámoslo cambiando la unidad (kg → gramos).

In [ ]:
kg = [p['kg'] for p in pedidos]
valor = [op.prioridad(p) for p in pedidos]
cap_kg = S['capacidad_kg']['VAN']
w = [max(1, round(x)) for x in kg]
n = 14
bf = reg.medir(runs, op.fuente, 'mochila', 'fuerza-bruta', n, True, lambda: L.knapsack_bruteforce(w[:n], valor[:n], cap_kg), valor_de=lambda r: r[0])
dp = reg.medir(runs, op.fuente, 'mochila', 'dp', n, True, lambda: L.knapsack_dp(w[:n], valor[:n], cap_kg), valor_de=lambda r: r[0])
gr = reg.medir(runs, op.fuente, 'mochila', 'voraz-densidad', n, False, lambda: L.knapsack_greedy(w[:n], valor[:n], cap_kg), optimo=dp[0], valor_de=lambda r: r[0])
assert bf[0] == dp[0] and L.verify_knapsack(w[:n], cap_kg, dp[1])
print(f'VAN {cap_kg} kg, {n} pedidos: óptimo={dp[0]:.0f} (pedidos {dp[1]}), voraz={gr[0]:.0f}')
tiempos = []
for escala, unidad in ((1, 'kg'), (10, '100 g'), (100, '10 g'), (1000, 'g')):
    wi = [max(1, round(x*escala)) for x in kg[:n]]
    t0 = time.perf_counter(); L.knapsack_dp(wi, valor[:n], cap_kg*escala); tiempos.append((unidad, round(time.perf_counter()-t0, 4)))
display(pd.DataFrame(tiempos, columns=['unidad de peso', 'segundos DP']))

**Lectura.** La mochila es NP-completa, pero con pesos "pequeños" la DP es rápida. Al pedir más precisión (gramos) el tiempo crece **aunque el problema sea el mismo**: eso es pseudo-polinomial. En la práctica: ¿cuánta precisión de peso necesita el negocio?

## 5. Empaque (bin packing): ¿cuántos viajes hacen falta? **NP-difícil**

Heurística **First-Fit Decreasing** con garantía $FFD \le \frac{11}{9}OPT + \frac{6}{9}$. Exacto con retroceso, solo para pocos pedidos.

In [ ]:
m = 12
pesos = [min(x, cap_kg) for x in kg[:m]]
ffd = reg.medir(runs, op.fuente, 'empaque', 'ffd', m, False, lambda: L.first_fit_decreasing(pesos, cap_kg), valor_de=len)
ex = reg.medir(runs, op.fuente, 'empaque', 'exacto', m, True, lambda: L.bin_packing_exact(pesos, cap_kg), valor_de=len)
runs[-2].optimo = len(ex); runs[-2].brecha_pct = round(100*(len(ffd)-len(ex))/len(ex), 2)
assert L.verify_bins(pesos, cap_kg, ex) and L.verify_bins(pesos, cap_kg, ffd)
print(f'{m} pedidos, VAN de {cap_kg} kg → FFD: {len(ffd)} viajes | óptimo: {len(ex)} viajes | cota teórica de FFD: {11/9*len(ex)+6/9:.2f}')
print('Carga mínima teórica (suma/cap):', round(sum(pesos)/cap_kg, 2), 'viajes')

## 6. Plazos y SLA: la ruta **más corta** no siempre cumple

Cada pedido tiene un plazo: el tiempo restante hasta su promesa (`promised_at − now()`), con un colchón (`margen_plazo`). Pregunta de decisión: ¿hay un recorrido que cumpla **todos** los plazos? Verificar es lineal; encontrarlo es NP-difícil (TSP con plazos). Comparamos tres recorridos: el de **mínima distancia** (Held-Karp), **plazo más próximo primero** (EDF, heurística) y el **mejor que cumple todos** (DP sobre subconjuntos).

In [ ]:
k = 8
sub = [[D[a][b] for b in range(k+1)] for a in range(k+1)]
dl = [p['restante_min'] * S['margen_plazo'] for p in pedidos[:k]]
t_hk, c_hk = tsp.held_karp(sub); t_hk = [x-1 for x in t_hk]
edf = reg.medir(runs, op.fuente, 'plazos', 'edf', k, False, lambda: L.earliest_deadline_first(sub, dl),
                valor_de=lambda r: tsp.tour_cost(sub, [x+1 for x in r]))
best = reg.medir(runs, op.fuente, 'plazos', 'dp-plazos', k, True, lambda: L.best_with_deadlines(sub, dl, MPC), valor_de=lambda r: r[1] if r else None)
def tarde(tour):
    return sum(a > dl[s] for a, s in zip(L.arrival_times(sub, tour, MPC), tour))
filas = [('mínima distancia (Held-Karp)', c_hk, tarde(t_hk)),
         ('plazo más próximo (EDF)', tsp.tour_cost(sub, [x+1 for x in edf]), tarde(edf))]
if best: filas.append(('mejor que cumple todo (DP)', best[1], tarde(best[0])))
display(pd.DataFrame(filas, columns=['recorrido', 'costo del ciclo', 'pedidos fuera de plazo']))
if not best: print('Ningún recorrido cumple todos los plazos: hay que renegociar o dividir el viaje.')

**Lectura estratégica.** Minimizar distancia **no** es minimizar incumplimientos. Cumplir el SLA puede costar más ruta; la pregunta gerencial es cuánto se paga por cumplir (costo del ciclo extra) frente a la penalidad por incumplir. Es la misma conclusión del laboratorio de decisión, ahora con su explicación teórica.

## 7. Asignar pedidos a varios robots: **NP-difícil** (parecido a empaque + TSP)

Cada robot tiene presupuesto de batería (100 % − reserva 20 % = 80 unidades de costo, supuesto del lab). Exacto: probar las $k^n$ asignaciones; heurístico: inserción más barata. Mide dónde el exacto deja de ser viable.

In [ ]:
budgets = [80, 80, 80]
filas = []
for n in range(4, 9):
    sub = [[D[a][b] for b in range(n+1)] for a in range(n+1)]
    ex = reg.medir(runs, op.fuente, 'asignacion', 'exacto', n, True, lambda: L.assign_exact(sub, budgets), valor_de=lambda r: r[1] if r else None)
    gr = reg.medir(runs, op.fuente, 'asignacion', 'insercion-barata', n, False, lambda: L.assign_greedy(sub, budgets),
                   optimo=ex[1] if ex else None, valor_de=lambda r: r[1] if r else None)
    assert ex is None or L.verify_assignment(sub, ex[0], budgets)
    filas.append({'n': n, 'asignaciones k^n': len(budgets)**n, 'óptimo': ex[1] if ex else None, 'voraz': gr[1] if gr else None,
                  's exacto': runs[-2].segundos, 's voraz': runs[-1].segundos})
display(pd.DataFrame(filas))

## 8. Cobertura (set cover): ¿qué transportistas contratar para cubrir todas las zonas con buen servicio?

Datos: `dw.fact_shipment` por transportista y zona; "cubre" una zona si su OTIF ≥ `otif_minimo_cobertura` con suficientes envíos. **NP-difícil**; el voraz da factor $H(n)\approx\ln n+1$.

In [ ]:
universo = op.zonas
cob = {k: v for k, v in op.cobertura.items() if v}
try:
    ex = reg.medir(runs, op.fuente, 'cobertura', 'exacto', len(cob), True, lambda: L.set_cover_exact(universo, cob), valor_de=len)
    gr = reg.medir(runs, op.fuente, 'cobertura', 'voraz', len(cob), False, lambda: L.set_cover_greedy(universo, cob), optimo=len(ex), valor_de=len)
    assert L.verify_set_cover(universo, cob, ex) and L.verify_set_cover(universo, cob, gr)
    print('Zonas:', sorted(universo)); print('Exacto :', ex); print('Voraz  :', gr)
    print('Cobertura por transportista:', {k: sorted(v) for k, v in cob.items()})
except ValueError as e:
    print('No se pueden cubrir todas las zonas con el OTIF exigido:', e)

## 9. Síntesis para la gerencia

Con lo medido arriba: qué garantía se puede prometer en cada problema y hasta qué tamaño.

**Mire la columna `brecha_max_pct`:** si las heurísticas quedan cerca de 0 en estos tamaños, el valor del algoritmo exacto no es *ahorrar* sino **poder afirmar** que no hay nada mejor (la garantía). Con instancias más grandes o más apretadas, esa brecha crece.

In [ ]:
res = pd.DataFrame([x.__dict__ for x in runs])
res = res[res.algoritmo != 'dijkstra-matriz']
resumen = res.groupby(['problema', 'algoritmo']).agg(n_max=('n', 'max'), seg_max=('segundos', 'max'),
                                                      brecha_max_pct=('brecha_pct', 'max'), exacto=('exacto', 'first')).round(4).reset_index()
display(resumen)

| Problema | Clase | Certificado (verificable en tiempo polinomial) | Exacto | Heurística y garantía | Qué promete el negocio |
|---|---|---|---|---|---|
| Distancias en el plano | **P** | — | Dijkstra | — | Respuesta inmediata |
| Recorrido de un viaje | NP-completo | El orden de paradas | Held-Karp (≤ 12–15) | NN/2-opt, **sin garantía** | Óptimo hasta ~12 paradas; más allá, "buena ruta", no "óptima" |
| Mochila (qué pedidos suben) | NP-completo (pseudo-poli.) | El subconjunto | DP en $O(nW)$ | Voraz sin cota fija | Óptimo si el peso se mide con poca precisión |
| Empaque (nº de viajes) | NP-difícil | La partición en viajes | Retroceso (≤ ~14) | FFD ≤ 11/9·OPT + 6/9 | Cota de calidad conocida |
| Asignación a robots | NP-difícil | Las rutas por robot | $k^n$ (≤ 8–9) | Inserción barata, sin garantía | Factible, sin garantía de mínimo |
| Plazos / SLA | NP-difícil | El recorrido | DP sobre subconjuntos | EDF, sin garantía | Se puede **verificar** el cumplimiento, no siempre **lograrlo** |
| Cobertura de zonas | NP-difícil | Los transportistas elegidos | Combinaciones | Voraz $H(n)$ | Cota logarítmica |

## 10. Guardar los resultados (Postgres y Data Lake) para verlos en Grafana

Las corridas se registran en `teoria.solver_run` (Postgres del lakehouse) y como CSV en `lake-curated/teoria/solver_runs.csv` (MinIO). El tablero `teoria_solver.json` las muestra. Si no hay base, la celda lo avisa y no falla.

In [ ]:
ok_pg = reg.guardar_postgres(runs)
ok_lake = reg.guardar_lake(runs)
print(f'{len(runs)} corridas | Postgres: {"guardado" if ok_pg else "no disponible"} | Data Lake: {"guardado" if ok_lake else "no disponible"}')

## 11. Preguntas para la exposición
1. Para el recorrido de un viaje, ¿qué parte es P y qué parte es NP-completa? ¿Qué es el certificado?
2. ¿Por qué la mochila es NP-completa y aun así la DP fue rápida? ¿Qué cambió al pasar de kg a gramos?
3. La heurística de empaque tiene garantía y la de asignación no: ¿qué le prometerían a un cliente en cada caso?
4. Con los plazos reales, ¿cuánto más cuesta cumplir el SLA que minimizar la distancia? ¿Compensa frente a la penalidad?
5. Si la flota crece 10× y los viajes pasan de 8 a 40 paradas, ¿qué algoritmo deja de ser viable y qué sustituirían?
6. ¿Qué parte de este análisis depende de supuestos (coordenadas, capacidades, margen de plazo) y no de los datos?